# Customer Churn Intelligence — DummyClassifier Baseline

## Objective

Establish the **most-frequent-class** dummy baseline before training real models. Evaluate on the **validation set only** — the final test set is not used.

**Stage:** Step 10 — Dummy baseline (no Logistic Regression, SMOTE, threshold tuning, SHAP, or test-set evaluation).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path("..").resolve()
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_PATH = REPORTS_DIR / "model_comparison.csv"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_split import load_split_from_manifest
from src.preprocessing import build_preprocessor

## 1. Load Train / Validation Splits

Test data is intentionally excluded from this notebook.

In [ ]:
split = load_split_from_manifest()

X_train, y_train_raw = split.X_train, split.y_train
X_val, y_val_raw = split.X_val, split.y_val

# Binary encoding for metrics (1 = Churn Yes)
y_train = (y_train_raw == "Yes").astype(int)
y_val = (y_val_raw == "Yes").astype(int)

print(f"Train rows: {len(X_train):,}")
print(f"Validation rows: {len(X_val):,}")
print(f"Validation churn prevalence: {y_val.mean():.2%}")

## 2. Build Pipeline (Preprocessor + DummyClassifier)

Preprocessor is fitted inside the pipeline on **training data only** when `pipe.fit(X_train, y_train)` is called.

In [ ]:
dummy_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]
)

dummy_pipeline.fit(X_train, y_train)
print("Pipeline fitted on training data.")

## 3. Predict on Validation Set

In [ ]:
y_val_pred = dummy_pipeline.predict(X_val)
y_val_proba = dummy_pipeline.predict_proba(X_val)[:, 1]

prediction_summary = pd.DataFrame(
    {
        "Metric": [
            "Actual churn (Yes) count",
            "Actual churn prevalence",
            "Predicted churn (Yes) count",
            "Predicted churn rate",
        ],
        "Value": [
            int(y_val.sum()),
            f"{y_val.mean():.2%}",
            int(y_val_pred.sum()),
            f"{y_val_pred.mean():.2%}",
        ],
    }
)
prediction_summary

## 4. Validation Metrics

In [ ]:
metrics = {
    "Accuracy": accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred, zero_division=0),
    "Recall": recall_score(y_val, y_val_pred, zero_division=0),
    "F1": f1_score(y_val, y_val_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_val, y_val_proba),
    "PR_AUC": average_precision_score(y_val, y_val_proba),
}

metrics_df = pd.DataFrame([metrics]).T.rename(columns={0: "DummyClassifier"})
metrics_df

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"],
)
print("Confusion Matrix (Validation):")
cm_df

## 5. Why Accuracy Is Misleading Here

The validation churn rate is ~**26.5%**. A dummy model that always predicts **No churn** achieves accuracy ≈ **73.5%** — equal to the majority-class proportion — while catching **zero** actual churners (Recall = 0).

For imbalanced churn problems:
- **Accuracy** rewards always guessing the majority class.
- **Recall** measures how many churners we identify (critical for retention campaigns).
- **Precision** measures how many flagged customers truly churn.
- **PR-AUC** is especially informative when the positive class is minority.

Any useful model must beat this baseline on **Recall, F1, ROC-AUC, and PR-AUC**, not accuracy alone.

## 6. Model Comparison Table

In [ ]:
comparison_row = {
    "Model": "DummyClassifier",
    "Accuracy": round(metrics["Accuracy"], 4),
    "Precision": round(metrics["Precision"], 4),
    "Recall": round(metrics["Recall"], 4),
    "F1": round(metrics["F1"], 4),
    "ROC_AUC": round(metrics["ROC_AUC"], 4),
    "PR_AUC": round(metrics["PR_AUC"], 4),
}

if COMPARISON_PATH.exists():
    comparison_df = pd.read_csv(COMPARISON_PATH)
    comparison_df = comparison_df[comparison_df["Model"] != "DummyClassifier"]
    comparison_df = pd.concat([comparison_df, pd.DataFrame([comparison_row])], ignore_index=True)
else:
    comparison_df = pd.DataFrame([comparison_row])

comparison_df = comparison_df[
    ["Model", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]
]
comparison_df.to_csv(COMPARISON_PATH, index=False)

print(f"Saved: {COMPARISON_PATH}")
comparison_df

## Baseline Summary

- **Model:** `DummyClassifier(strategy='most_frequent')` inside preprocessing `Pipeline`
- **Training:** 4,930 rows | **Validation:** 1,056 rows | **Test:** not used
- **Behavior:** Always predicts **No churn** (majority class)
- **Accuracy ~73.5%** mirrors majority-class rate but **Recall = 0** — useless for retention
- **ROC-AUC = 0.5** (no ranking ability) | **PR-AUC ≈ 26.5%** (equals random baseline at prevalence)

**Next step (not performed here):** Logistic Regression baseline.